# 백엔드-3 (FCT FAIL Step 반복 분석) — 단발성 Notebook (DAY + NIGHT)

## 확정 규칙
- 원천: `a2_fct_table.fct_table`
- 필터: `result='FAIL'`
- PN: `barcode_information`의 **18번째 문자**로 `g_production_film.remark_info` 매핑, 없으면 `UNKNOWN`
- 기준 키: **(barcode_information + step_description)**
- bucket:
  - 1회 FAIL → `1회 FAIL_step_description`
  - 2회 반복 FAIL → `2회 반복_FAIL_step_description`
  - 3회 이상 반복 FAIL → `3회 이상 반복_FAIL_step_description`
- count 정의(중요): 각 bucket에 속하는 **(barcode_information, step_description) 조합 개수**
  - 예) 2회 반복 FAIL 조합이 3개면 count=3
- 시간 윈도우(KST):
  - day: D 08:30:00 ~ 20:29:59 (포함)
  - night: D 20:30:00 ~ 23:59:59 (포함) + D+1 00:00:00 ~ 08:29:59 (포함)
- 산출물: DataFrame 출력 (저장/UPSERT 없음)


## 0) 입력 파라미터

In [1]:
from __future__ import annotations

from datetime import datetime, date, timedelta
from zoneinfo import ZoneInfo

KST = ZoneInfo('Asia/Seoul')

# 기준 생산일
PROD_DAY = '20260130'  # YYYYMMDD

# 계산할 shift 목록
SHIFTS = ['day', 'night']
assert all(s in ('day','night') for s in SHIFTS)


## 1) DB 설정

In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

DB_CONFIG = {
    'host': '100.105.75.47',
    'port': 5432,
    'dbname': 'postgres',
    'user': 'postgres',
    'password': '',
}

os.environ.setdefault('PGCLIENTENCODING', 'UTF8')
os.environ.setdefault('PGOPTIONS', '-c client_encoding=UTF8')

def make_engine() -> Engine:
    url = (
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    )
    return create_engine(
        url,
        pool_size=1,
        max_overflow=0,
        pool_pre_ping=True,
        pool_recycle=1800,
    )

engine = make_engine()
print('engine ready')


engine ready


## 2) 유틸

In [3]:
def yyyymmdd_to_date(s: str) -> date:
    return date(int(s[0:4]), int(s[4:6]), int(s[6:8]))

def next_day_yyyymmdd(prod_day: str) -> str:
    d = yyyymmdd_to_date(prod_day)
    return (d + timedelta(days=1)).strftime('%Y%m%d')


## 3) (barcode+step) 기준 FAIL 횟수(pair_cnt) 추출 SQL

In [4]:
SRC_SCHEMA = 'a2_fct_table'
SRC_TABLE  = 'fct_table'
REMARK_SCHEMA = 'g_production_film'
REMARK_TABLE  = 'remark_info'

PAIR_SQL = f"""
WITH base AS (
    SELECT
        :prod_day AS prod_day,
        :shift_type AS shift_type,
        COALESCE(ri.pn, 'UNKNOWN') AS pn,
        ft.barcode_information,
        ft.step_description,
        ft.end_day,
        LPAD(ft.end_time, 6, '0') AS end_time_norm
    FROM {SRC_SCHEMA}.{SRC_TABLE} ft
    LEFT JOIN {REMARK_SCHEMA}.{REMARK_TABLE} ri
        ON ri.barcode_information = SUBSTRING(ft.barcode_information FROM 18 FOR 1)
    WHERE
        ft.result = 'FAIL'
        AND (
            (:shift_type = 'day'
             AND ft.end_day = :prod_day
             AND LPAD(ft.end_time, 6, '0') BETWEEN '083000' AND '202959')
            OR
            (:shift_type = 'night'
             AND (
                 (ft.end_day = :prod_day AND LPAD(ft.end_time, 6, '0') BETWEEN '203000' AND '235959')
                 OR
                 (ft.end_day = :next_day AND LPAD(ft.end_time, 6, '0') BETWEEN '000000' AND '082959')
             )
            )
        )
),
pair_cnt AS (
    SELECT
        prod_day,
        shift_type,
        pn,
        barcode_information,
        step_description,
        COUNT(*) AS fail_cnt
    FROM base
    GROUP BY prod_day, shift_type, pn, barcode_information, step_description
)
SELECT * FROM pair_cnt;
"""


## 4) bucket별 DF 생성 (count = 조합 개수)

In [5]:
UPDATED_AT = datetime.now(KST)

COL_1 = '1회 FAIL_step_description'
COL_2 = '2회 반복_FAIL_step_description'
COL_3 = '3회 이상 반복_FAIL_step_description'

def bucket_df(df_pairs: pd.DataFrame, bucket: str) -> pd.DataFrame:
    if bucket == '1st':
        df = df_pairs[df_pairs['fail_cnt'] == 1].copy()
        step_col = COL_1
    elif bucket == '2nd':
        df = df_pairs[df_pairs['fail_cnt'] == 2].copy()
        step_col = COL_2
    elif bucket == '3rd_over':
        df = df_pairs[df_pairs['fail_cnt'] >= 3].copy()
        step_col = COL_3
    else:
        raise ValueError(bucket)

    if df.empty:
        return pd.DataFrame(columns=['prod_day','shift_type','pn',step_col,'count','updated_at'])

    out = (
        df.groupby(['prod_day','shift_type','pn','step_description'], as_index=False)
          .agg(count=('fail_cnt','size'))  # ✅ 조합 개수
    )
    out = out.rename(columns={'step_description': step_col})
    out['updated_at'] = UPDATED_AT
    out = out.sort_values(['count','pn'], ascending=[False, True]).reset_index(drop=True)
    return out

def run_one(prod_day: str, shift_type: str) -> tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame,pd.DataFrame]:
    nd = next_day_yyyymmdd(prod_day)
    params = {'prod_day': prod_day, 'shift_type': shift_type, 'next_day': nd}
    with engine.connect() as conn:
        df_pairs = pd.read_sql(text(PAIR_SQL), conn, params=params)
    df1 = bucket_df(df_pairs, '1st')
    df2 = bucket_df(df_pairs, '2nd')
    df3 = bucket_df(df_pairs, '3rd_over')
    return df_pairs, df1, df2, df3


## 5) PROD_DAY 기준 DAY / NIGHT 계산 및 출력

In [6]:
results = {}

for sh in SHIFTS:
    df_pairs, df1, df2, df3 = run_one(PROD_DAY, sh)
    results[sh] = {'pairs': df_pairs, 'df1': df1, 'df2': df2, 'df3': df3}
    print(f"\n=== {PROD_DAY} / {sh} ===")
    if len(df_pairs):
        display(df_pairs['fail_cnt'].value_counts().sort_index())
        print('max fail_cnt =', int(df_pairs['fail_cnt'].max()))
    else:
        print('no FAIL rows in window')
    print('1회 rows:', len(df1), '| 2회 rows:', len(df2), '| 3회+ rows:', len(df3))
    display(df1)
    display(df2)
    display(df3)



=== 20260130 / day ===


fail_cnt
1    78
2     3
Name: count, dtype: int64

max fail_cnt = 2
1회 rows: 8 | 2회 rows: 1 | 3회+ rows: 0


,prod_day,shift_type,pn,1회 FAIL_step_description,count,updated_at
0,20260130,day,35930927,1.34_Test_VUSB_Type-C_A(ELoad2=1.35A)vol,56,2026-01-31 13:11:42.237670+09:00
1,20260130,day,35930927,1.02_Test_USB2_error(Type-C_A_side),7,2026-01-31 13:11:42.237670+09:00
2,20260130,day,35930928,1.34_Test_VUSB_Type-C_A(ELoad2=1.35A)vol,5,2026-01-31 13:11:42.237670+09:00
3,20260130,day,35643010,1.05 Test DIM-GND(ohm),4,2026-01-31 13:11:42.237670+09:00
4,20260130,day,35930928,1.17 Test DIM-GND(ohm),3,2026-01-31 13:11:42.237670+09:00
5,20260130,day,35930927,1.28_Test_IELoad2_Type-C(B_side),1,2026-01-31 13:11:42.237670+09:00
6,20260130,day,35930927,1.32_PD Negotiation SET PDO4,1,2026-01-31 13:11:42.237670+09:00
7,20260130,day,35930928,1.28_Test_IELoad2_Type-C(B_side),1,2026-01-31 13:11:42.237670+09:00


,prod_day,shift_type,pn,2회 반복_FAIL_step_description,count,updated_at
0,20260130,day,35930927,1.02_Test_USB2_error(Type-C_A_side),3,2026-01-31 13:11:42.237670+09:00


...

fail_cnt
1    159
2      7
Name: count, dtype: int64

max fail_cnt = 2
1회 rows: 16 | 2회 rows: 3 | 3회+ rows: 0


,prod_day,shift_type,pn,1회 FAIL_step_description,count,updated_at
0,20260130,night,35930928,1.34_Test_VUSB_Type-C_A(ELoad2=1.35A)vol,85,2026-01-31 13:11:42.237670+09:00
1,20260130,night,35930928,1.28_Test_IELoad2_Type-C(B_side),40,2026-01-31 13:11:42.237670+09:00
2,20260130,night,35930928,1.02_Test_USB2_error(Type-C_A_side),6,2026-01-31 13:11:42.237670+09:00
3,20260130,night,35930928,1.14 Profile Count Check,5,2026-01-31 13:11:42.237670+09:00
4,20260130,night,35930928,1.31_Test_Type-A(ELoad1=2.4A),5,2026-01-31 13:11:42.237670+09:00
5,20260130,night,35930928,1.17 Test DIM-GND(ohm),3,2026-01-31 13:11:42.237670+09:00
6,20260130,night,35643010,1.05 Test DIM-GND(ohm),2,2026-01-31 13:11:42.237670+09:00
7,20260130,night,35930928,1.03_Test USB2 benchmark.maxrd(Mbit/s),2,2026-01-31 13:11:42.237670+09:00
8,20260130,night,35930928,1.04 Test USB2 benchmark.maxwr(Mbit/s),2,2026-01-31 13:11:42.237670+09:00
9,20260130,night,35930928,1.13_Test_Carplay_Type-A,2,2026-01-31 13:11:42.237670+09:00


,prod_day,shift_type,pn,2회 반복_FAIL_step_description,count,updated_at
0,20260130,night,35930928,1.02_Test_USB2_error(Type-C_A_side),4,2026-01-31 13:11:42.237670+09:00
1,20260130,night,35930928,1.13_Test_Carplay_Type-A,2,2026-01-31 13:11:42.237670+09:00
2,20260130,night,35643010,1.05 Test DIM-GND(ohm),1,2026-01-31 13:11:42.237670+09:00


,prod_day,shift_type,pn,3회 이상 반복_FAIL_step_description,count,updated_at


In [9]:
# ============================================
# 7) 결과 DF를 i_daily_report 스키마에 저장 (DAY/NIGHT, 1/2/3+)
# - 스키마/테이블 자동 생성(없으면 생성)
# - UPSERT(중복 키 기준 갱신)
# - 컬럼은 TEXT, updated_at은 timestamptz
# - ✅ 한글/공백 컬럼명 대응: bind param은 v0,v1..로 안전 매핑
# ============================================

from sqlalchemy import text
from sqlalchemy.engine import Engine
import pandas as pd

SAVE_SCHEMA = "i_daily_report"

DAY_T1 = "c_1time_step_decription_day_daily"
DAY_T2 = "c_2time_step_decription_day_daily"
DAY_T3 = "c_3time_over_step_decription_day_daily"

NIGHT_T1 = "c_1time_step_decription_night_daily"
NIGHT_T2 = "c_2time_step_decription_night_daily"
NIGHT_T3 = "c_3time_over_step_decription_night_daily"


def ensure_schema(engine: Engine, schema: str):
    with engine.begin() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema};"))


def ensure_table(engine: Engine, schema: str, table: str, columns: list[str], key_cols: list[str]):
    """
    columns: DF 컬럼명 리스트 (DB에 TEXT로 생성, updated_at만 timestamptz)
    key_cols: UNIQUE key 컬럼 리스트
    """
    ddl_cols = []
    for c in columns:
        if c == "updated_at":
            ddl_cols.append(f'"{c}" timestamptz')
        else:
            ddl_cols.append(f'"{c}" text')

    ddl_cols_sql = ",\n  ".join(ddl_cols)
    key_sql = ", ".join([f'"{c}"' for c in key_cols])

    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {schema}.{table} (
      {ddl_cols_sql},
      CONSTRAINT {table}__uk UNIQUE ({key_sql})
    );
    """
    with engine.begin() as conn:
        conn.execute(text(create_sql))


def normalize_df_for_db(df: pd.DataFrame) -> pd.DataFrame:
    """
    - updated_at 제외한 모든 컬럼을 string으로 캐스팅
    - NaN -> None
    """
    if df is None:
        return df
    out = df.copy()
    for c in out.columns:
        if c == "updated_at":
            continue
        out[c] = out[c].astype("string")
    out = out.where(pd.notnull(out), None)
    return out


def upsert_df(engine: Engine, schema: str, table: str, df: pd.DataFrame, key_cols: list[str]):
    """
    df를 schema.table로 UPSERT.
    - key_cols로 UNIQUE 충돌 시 UPDATE
    - ✅ 한글/공백 컬럼명 대응: VALUES(:v0,:v1...) 형태로 바인딩
    """
    if df is None or df.empty:
        print(f"[SKIP] {schema}.{table}: empty df")
        return

    cols = list(df.columns)

    # 컬럼명은 DB에서는 그대로 쓰되(따옴표),
    # bind param은 v0,v1... 로 안전하게 분리
    pkeys = [f"v{i}" for i in range(len(cols))]

    col_sql = ", ".join([f'"{c}"' for c in cols])
    val_sql = ", ".join([f":{k}" for k in pkeys])

    key_sql = ", ".join([f'"{c}"' for c in key_cols])

    # UPDATE SET: key 제외 나머지 컬럼 갱신
    set_sql = ", ".join([f'"{c}" = EXCLUDED."{c}"' for c in cols if c not in key_cols])

    sql = f"""
    INSERT INTO {schema}.{table} ({col_sql})
    VALUES ({val_sql})
    ON CONFLICT ({key_sql})
    DO UPDATE SET
      {set_sql}
    ;
    """

    # executemany용 레코드: {'v0':..., 'v1':...} 형태로 통일
    recs_raw = df.to_dict(orient="records")
    records = []
    for r in recs_raw:
        records.append({k: r[c] for c, k in zip(cols, pkeys)})

    with engine.begin() as conn:
        conn.execute(text(sql), records)

    print(f"[OK] upserted {len(df)} rows -> {schema}.{table}")


def key_cols_for(df: pd.DataFrame) -> list[str]:
    # step description 컬럼 찾기(한글 prefix라도 suffix는 step_description)
    step_cols = [c for c in df.columns if str(c).endswith("step_description")]
    if not step_cols:
        raise RuntimeError(f"step_description 컬럼을 찾을 수 없음: {list(df.columns)}")
    step_col = step_cols[0]
    return ["prod_day", "shift_type", "pn", step_col]


def create_and_upsert(table: str, df: pd.DataFrame):
    if df is None or df.empty:
        print(f"[SKIP] {SAVE_SCHEMA}.{table}: empty df")
        return
    df = normalize_df_for_db(df)
    cols = list(df.columns)
    keys = key_cols_for(df)
    ensure_table(engine, SAVE_SCHEMA, table, cols, keys)
    upsert_df(engine, SAVE_SCHEMA, table, df, keys)


# =========================
# 저장 대상 DF (앞 셀에서 results dict 생성되어 있어야 함)
# =========================
df_day_1   = results["day"]["df1"]
df_day_2   = results["day"]["df2"]
df_day_3   = results["day"]["df3"]

df_night_1 = results["night"]["df1"]
df_night_2 = results["night"]["df2"]
df_night_3 = results["night"]["df3"]

# =========================
# 실행
# =========================
ensure_schema(engine, SAVE_SCHEMA)

# ---- DAY ----
create_and_upsert(DAY_T1, df_day_1)
create_and_upsert(DAY_T2, df_day_2)
create_and_upsert(DAY_T3, df_day_3)

# ---- NIGHT ----
create_and_upsert(NIGHT_T1, df_night_1)
create_and_upsert(NIGHT_T2, df_night_2)
create_and_upsert(NIGHT_T3, df_night_3)

[OK] upserted 8 rows -> i_daily_report.c_1time_step_decription_day_daily
[OK] upserted 1 rows -> i_daily_report.c_2time_step_decription_day_daily
[SKIP] i_daily_report.c_3time_over_step_decription_day_daily: empty df
[OK] upserted 16 rows -> i_daily_report.c_1time_step_decription_night_daily
[OK] upserted 3 rows -> i_daily_report.c_2time_step_decription_night_daily
[SKIP] i_daily_report.c_3time_over_step_decription_night_daily: empty df
